In [25]:
import json
import textwrap
import copy
import logging
from tqdm import tqdm


logger = logging.getLogger(__name__)


In [2]:
import pickle
from pathlib import Path

test_set = pickle.load(open(Path.cwd().parent / "local_data" / "test_set.pkl", "rb"))
texts = [data["text"] for data in test_set][35:45]
pmids = [data["pmid"] for data in test_set][35:45]


## LAYER 1: 
### Decompose input expression into tree, in terms of epistemic mode and propositional logic operations

INPUT: A string of any natural language expression.

OUTPUT: An expression tree decomposed under epistemic modality and propositional logic operations.

In [3]:
from l1_decompose_expr import *


In [4]:
INPUT = (texts, pmids)


In [5]:
OUTPUT = {}
for i, (text, pmid) in tqdm(enumerate(zip(*INPUT)), total=len(INPUT[0])):
    expr_tree = expression_parse_recursive(
        text,
        parse_surface_epistemic_mode=parse_surface_epistemic_mode,
        parse_surface_logical_grouping=parse_surface_logical_grouping,
        parse_surface_logical_unary=parse_surface_logical_unary,
        epistemic_mode_schema=epistemic_mode_schema,
        logical_grouping_schema=logical_grouping_schema,
        logical_unary_schema=logical_unary_schema,
        max_depth=5
    )
    OUTPUT[i] = {
        "text": text,
        "pmid": pmid,
        "expr_tree": expr_tree,
    }


 50%|█████     | 5/10 [00:46<00:46,  9.35s/it]Max depth 5 exceeded at epistemic level. Returning fallback.
Max depth 5 exceeded at epistemic level. Returning fallback.
100%|██████████| 10/10 [01:44<00:00, 10.46s/it]


In [6]:
print("OUTPUT:")
print(json.dumps(OUTPUT, indent=2))


OUTPUT:
{
  "0": {
    "text": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity, leading to increased cell proliferation and S-phase population.",
    "pmid": "26473517",
    "expr_tree": {
      "DECLARATIVE": {
        "IDENTITY": {
          "AND": [
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased HaCaT cell proliferation"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased S-phase population in HaCaT cells"
                }
              }
            }
          ]
        }
      }
    }
  },
  "1": {
    "text": 

## LAYER 2: 
### Collect and filter from all LEAF nodes of tree from LAYER 1, to get all valid statements that are fully reduced under relation-level coordination.
INPUT: An expression tree decomposed under epistemic modality and propositional logic operations.

OUTPUT: A list of statements fully reduced under relation-level coordination. 

In [7]:
from l2_collect_valid_statements import *


In [8]:
for key, value in OUTPUT.items():
    valid_statements = valid_from_exp_tree(value['expr_tree'])
    OUTPUT[key]["valid_statements"] = valid_statements


In [9]:
print("OUTPUT:")
print(json.dumps(OUTPUT, indent=2))


OUTPUT:
{
  "0": {
    "text": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity, leading to increased cell proliferation and S-phase population.",
    "pmid": "26473517",
    "expr_tree": {
      "DECLARATIVE": {
        "IDENTITY": {
          "AND": [
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased HaCaT cell proliferation"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased S-phase population in HaCaT cells"
                }
              }
            }
          ]
        }
      }
    },
    "valid_statements": [

## LAYER 3: 
### Decompose input statement string into a tree, in terms of n-ary relation components. 

INPUT: A string of statement which is fully reduced in terms of relation-level coordination. 

OUTPUT: A tree-structured dictionary representing the n-ary relation structure decomposition of an input statement, where each internal node corresponds to a relation and its associated semantic arguments.

In [10]:
from l3_decompose_statement import *


In [11]:
for key, item in tqdm(OUTPUT.items(), total=len(OUTPUT)):
    valid_statements = item['valid_statements']
    if len(valid_statements) == 0:
        continue
    else: 
        OUTPUT[key]["relationship_trees"] = {}
        for i, valid_statement in enumerate(valid_statements):
            try:
                relationship_tree = parse_relationship_recursive(
                    valid_statement,
                    parse_surface_relationship=parse_surface_relationship,
                    parse_surface_argument=parse_surface_argument,
                    relationship_schema=relationship_schema,
                    argument_schema=argument_schema,
                    depth=0,
                    max_depth=5
                )
            except Exception as e:
                logger.error(f"Error parsing statement: {valid_statement}")
                logger.error(e)
                relationship_tree = None
            OUTPUT[key]["relationship_trees"][i] = {
                "valid_statement": valid_statement,
                "relationship_tree": relationship_tree,
            }


100%|██████████| 10/10 [04:34<00:00, 27.41s/it]


In [12]:
print("OUTPUT:")
print(json.dumps(OUTPUT, indent=2))


OUTPUT:
{
  "0": {
    "text": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity, leading to increased cell proliferation and S-phase population.",
    "pmid": "26473517",
    "expr_tree": {
      "DECLARATIVE": {
        "IDENTITY": {
          "AND": [
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased HaCaT cell proliferation"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased S-phase population in HaCaT cells"
                }
              }
            }
          ]
        }
      }
    },
    "valid_statements": [

## LAYER 4: 
### Collect and filter from all STATEMENT nodes of tree from LAYER 3, to get all valid statements whose entities lack verb-like parts of speech.

INPUT: A statement tree/dict decomposed in terms of statement relationship components

OUTPUT: A list of relationship. Each relationship's argument is a non-verbal phrase.

In [13]:
from l4_collect_valid_relationships import extract_terminal_relationships


In [14]:
for key, item in OUTPUT.items():
    relationship_trees = item.get('relationship_trees', {})
    if not relationship_trees:
        continue
    collected_relationships = []
    for i, (tree_key, relationship_tree) in enumerate(relationship_trees.items()):
        valid_relationships = extract_terminal_relationships(relationship_tree['relationship_tree'])
        if key not in OUTPUT:
            OUTPUT[key] = {"relationship_trees": {}}
        if tree_key not in OUTPUT[key]["relationship_trees"]:
            OUTPUT[key]["relationship_trees"][tree_key] = {}

        OUTPUT[key]["relationship_trees"][tree_key]['valid_relationships'] = valid_relationships
        collected_relationships.extend(valid_relationships)
    OUTPUT[key]['collected_relationships'] = collected_relationships


In [15]:
print("OUTPUT:")
print(json.dumps(OUTPUT, indent=2))


OUTPUT:
{
  "0": {
    "text": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity, leading to increased cell proliferation and S-phase population.",
    "pmid": "26473517",
    "expr_tree": {
      "DECLARATIVE": {
        "IDENTITY": {
          "AND": [
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased HaCaT cell proliferation"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased S-phase population in HaCaT cells"
                }
              }
            }
          ]
        }
      }
    },
    "valid_statements": [

## LAYER 5: 
### Extract INDRA statement from each relationship from list of relationships collected from LAYER 5.

INPUT: A relation of form `{"relation": <string>, "arity": <int>, "arguments": <list of string>}`

OUTPUT: A JSON INDRA statement

In [16]:
from l5_extract_indra_stmt_from_relationship import *
from indra_gpt.configs import BaseConfig, PostProcessorConfig
from indra_gpt.processors import PostProcessor

base_config = BaseConfig({"grounding": True, "grounding_strategy": "gilda", "num_samples": 100})
postprocessing_config = PostProcessorConfig(base_config)
post_processor = PostProcessor(postprocessing_config)


INFO: [2025-04-04 17:01:13] indra.preassembler.grounding_mapper.disambiguate - INDRA DB is not available for text content retrieval for grounding disambiguation.


In [17]:
for key, item in tqdm(OUTPUT.items(), total=len(OUTPUT)):
    text = item['text']
    pmid = item['pmid']
    collected_relationships = item.get('collected_relationships', []) 

    json_stmts = []
    for relationship in collected_relationships:
        json_stmt = extract_indra_statement_from_relationship(str(relationship))
        json_stmts.append(json_stmt)
        # preprocessing block
        json_stmts = [post_processor.remove_empty_values(json_stmt) for json_stmt in json_stmts]
        json_stmts = [post_processor.update_evidence(stmt_json, text, pmid, postprocessing_config) 
                      for stmt_json in json_stmts] 
        json_stmts = [post_processor.add_evidence(stmt_json, str(relationship), pmid, postprocessing_config) 
                      for stmt_json in json_stmts] 
        for json_stmt in json_stmts:
            json_stmt.pop('id', None)
            json_stmt.pop('supports', None)
            json_stmt.pop('supported_by', None)
    OUTPUT[key]['json_stmts'] = json_stmts
    # Convert the json statements to raw INDRA statements.
    raw_stmts = post_processor.extract_stmts(json_stmts)
    OUTPUT[key]['raw_stmts'] = raw_stmts
    # Update the db_refs of the agents, because this is required for the grounding.
    for stmt in raw_stmts:
        try:
            for agent in stmt.agent_list():
                if agent and not agent.db_refs.get('TEXT', None):
                    agent.db_refs.update({'TEXT': agent.name})
        except Exception as e:
            logger.warning(f"Failed to update db_refs for agent: {stmt} — error: {e}")
    # Get preassembled statements.
    try:
        preassembled_stmts = post_processor.preassembly_pipeline(raw_stmts)
    except Exception as e: 
        logger.warning(e)
        preassembled_stmts = []
    OUTPUT[key]['preassembled_stmts'] = preassembled_stmts


  0%|          | 0/10 [00:00<?, ?it/s]INFO: [2025-04-04 17:01:13] indra.pipeline.pipeline - Running the pipeline
INFO: [2025-04-04 17:01:13] indra.pipeline.pipeline - Calling filter_no_hypothesis
INFO: [2025-04-04 17:01:13] indra.tools.assemble_corpus - Filtering 0 statements to no hypothesis...
INFO: [2025-04-04 17:01:13] indra.tools.assemble_corpus - 0 statements after filter...
INFO: [2025-04-04 17:01:13] indra.pipeline.pipeline - Calling filter_no_negated
INFO: [2025-04-04 17:01:13] indra.tools.assemble_corpus - Filtering 0 statements to not negated...
INFO: [2025-04-04 17:01:13] indra.tools.assemble_corpus - 0 statements after filter...
INFO: [2025-04-04 17:01:13] indra.pipeline.pipeline - Calling custom_ground_statements
INFO: [2025-04-04 17:01:13] indra_gpt.processors.post_processors - Grounding 0 statements with GILDA...
INFO: [2025-04-04 17:01:13] indra_gpt.processors.post_processors - Grounded 0 statements with GILDA.
INFO: [2025-04-04 17:01:13] indra.pipeline.pipeline - Call

In [18]:
def indra_serializer(obj):
    try:
        return str(obj)  # fallback to string if not serializable
    except Exception:
        return f"<non-serializable: {type(obj).__name__}>"


In [19]:
print("FINAL OUTPUT:")
print(json.dumps(OUTPUT, indent=2, default=indra_serializer))


FINAL OUTPUT:
{
  "0": {
    "text": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity, leading to increased cell proliferation and S-phase population.",
    "pmid": "26473517",
    "expr_tree": {
      "DECLARATIVE": {
        "IDENTITY": {
          "AND": [
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "NCSTN knockdown in HaCaT cells impaired \u03b3-secretase activity"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased HaCaT cell proliferation"
                }
              }
            },
            {
              "DECLARATIVE": {
                "IDENTITY": {
                  "ATOM": "\u03b3-secretase activity led to increased S-phase population in HaCaT cells"
                }
              }
            }
          ]
        }
      }
    },
    "valid_statemen

In [20]:
preassembled_stmts = []
for key, item in OUTPUT.items():
    preassembled_stmts.extend(item.get('preassembled_stmts', []))


In [ ]:
preassembled_stmts


[Activation(RAPGEF1(), JNK(), kinase)]

In [23]:
preassembled_stmts[0].to_json()


OrderedDict([('type', 'Activation'),
             ('subj',
              OrderedDict([('name', 'RAPGEF1'),
                           ('db_refs',
                            {'TEXT': 'C3G',
                             'HGNC': '4568',
                             'EGID': '2889',
                             'UP': 'Q13905'})])),
             ('obj',
              OrderedDict([('name', 'JNK'),
                           ('db_refs',
                            {'TEXT': 'JNK',
                             'FPLX': 'JNK',
                             'IP': 'IPR008351'})])),
             ('obj_activity', 'kinase'),
             ('belief', 0.81),
             ('evidence',
              [OrderedDict([('source_api', 'indra_gpt'),
                            ('pmid', '9430657'),
                            ('text',
                             'Ras-independent activation of JNK was supported by the finding that C3G-induced JNK activation was not inhibited by the dominant negative forms of Rac or 